# MEMORIA AI Prototype: Local LLM Evaluation

Deterministic 15-model local benchmark with per-model thinking configs, crash-safe JSON persistence, and post-run merge-based comparison.


## Environment Setup (CPU, MPS, CUDA)

Run this first in terminal:

```bash
conda activate myenv-django
python -m pip install -r requirements.txt
python -m pip install -U huggingface_hub accelerate
hf auth login
hf auth whoami
```

NVIDIA CUDA PyTorch install (Linux or Windows with NVIDIA GPU):

```bash
python -m pip uninstall -y torch torchvision torchaudio
python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
```

Compatibility:
- macOS Apple Silicon uses MPS
- NVIDIA Linux or Windows uses CUDA
- CPU fallback is supported

Optional Linux or NVIDIA extras (optional):

```bash
python -m pip install -U xformers
```


In [ ]:
from pathlib import Path
import os
import gc
import json
import time
import traceback
import tempfile
import shutil
import warnings
from datetime import datetime, timezone

import torch
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer
from openai import OpenAI

warnings.filterwarnings("ignore", message=r".*IProgress not found.*")
warnings.filterwarnings("ignore", message=r".*tied weights mapping and config for this model specifies to tie.*")
warnings.filterwarnings("ignore", message=r".*Some parameters are on the meta device because they were offloaded to the disk.*")

openaiClient = OpenAI()
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-5.1")


## Hugging Face Login Reminder

Before running model cells, make sure Hugging Face auth is active in `myenv-django`:

```bash
hf auth login
hf auth whoami
```

If authentication is missing, gated models (for example Gemma) will fail to load.


In [ ]:
if (Path.cwd() / "ai_prototype.ipynb").exists():
    NOTEBOOK_DIR = Path.cwd()
elif (Path.cwd() / "llm_test" / "ai_prototype.ipynb").exists():
    NOTEBOOK_DIR = Path.cwd() / "llm_test"
else:
    raise RuntimeError("Unable to determine notebook directory. Start from repo root or llm_test.")

REPO_ROOT = NOTEBOOK_DIR.parent
ENV_PATH = REPO_ROOT / ".env"
if not ENV_PATH.exists():
    raise RuntimeError(f"Missing .env file at {ENV_PATH}")
load_dotenv(ENV_PATH)

CACHE_ROOT = NOTEBOOK_DIR / "cache" / "huggingface-models"
RESULTS_DIR = NOTEBOOK_DIR / "results"
BY_MODEL_DIR = RESULTS_DIR / "by_model"
RUN_LOG_JSONL_PATH = RESULTS_DIR / "model_runs.jsonl"
MERGED_RESULTS_PATH = RESULTS_DIR / "merged_results.json"

CACHE_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BY_MODEL_DIR.mkdir(parents=True, exist_ok=True)

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"Notebook Dir: {NOTEBOOK_DIR}")
print(f"Device: {DEVICE}")
print(f"Cache Root: {CACHE_ROOT}")
print(f"Results Dir: {RESULTS_DIR}")


---
## Section 1: Model Registry and Per-Model Configs


In [ ]:
MODEL_REGISTRY = [
    {
        "modelId": "Qwen/Qwen3.5-0.8B",
        "family": "Qwen",
        "category": "Ultra-Light",
        "params": "0.8B",
        "version": "Qwen3.5",
        "license": "Apache-2.0"
    },
    {
        "modelId": "Qwen/Qwen3.5-2B",
        "family": "Qwen",
        "category": "Small",
        "params": "2B",
        "version": "Qwen3.5",
        "license": "Apache-2.0"
    },
    {
        "modelId": "Qwen/Qwen3.5-4B",
        "family": "Qwen",
        "category": "Medium",
        "params": "4B",
        "version": "Qwen3.5",
        "license": "Apache-2.0"
    },
    {
        "modelId": "Qwen/Qwen3.5-9B",
        "family": "Qwen",
        "category": "Large",
        "params": "9B",
        "version": "Qwen3.5",
        "license": "Apache-2.0"
    },
    {
        "modelId": "Qwen/Qwen3.5-27B",
        "family": "Qwen",
        "category": "Extra-Large",
        "params": "27B",
        "version": "Qwen3.5",
        "license": "Apache-2.0"
    },
    {
        "modelId": "google/gemma-3-1b-it",
        "family": "Gemma",
        "category": "Ultra-Light",
        "params": "1B",
        "version": "Gemma3",
        "license": "Gemma"
    },
    {
        "modelId": "google/gemma-2-2b-it",
        "family": "Gemma",
        "category": "Small",
        "params": "2B",
        "version": "Gemma2",
        "license": "Gemma"
    },
    {
        "modelId": "google/gemma-3-4b-it",
        "family": "Gemma",
        "category": "Medium",
        "params": "4B",
        "version": "Gemma3",
        "license": "Gemma"
    },
    {
        "modelId": "google/gemma-3-12b-it",
        "family": "Gemma",
        "category": "Large",
        "params": "12B",
        "version": "Gemma3",
        "license": "Gemma"
    },
    {
        "modelId": "google/gemma-3-27b-it",
        "family": "Gemma",
        "category": "Extra-Large",
        "params": "27B",
        "version": "Gemma3",
        "license": "Gemma"
    },
    {
        "modelId": "zai-org/glm-edge-1.5b-chat",
        "family": "GLM",
        "category": "Ultra-Light",
        "params": "1.5B",
        "version": "GLM-Edge",
        "license": "Other"
    },
    {
        "modelId": "zai-org/glm-edge-4b-chat",
        "family": "GLM",
        "category": "Small",
        "params": "4B",
        "version": "GLM-Edge",
        "license": "Other"
    },
    {
        "modelId": "zai-org/chatglm3-6b",
        "family": "GLM",
        "category": "Medium",
        "params": "6B",
        "version": "ChatGLM3",
        "license": "Other"
    },
    {
        "modelId": "zai-org/glm-4-9b-chat",
        "family": "GLM",
        "category": "Large",
        "params": "9.4B",
        "version": "GLM-4",
        "license": "Other"
    },
    {
        "modelId": "zai-org/GLM-5",
        "family": "GLM",
        "category": "Extra-Large",
        "params": "MoE",
        "version": "GLM-5",
        "license": "MIT"
    }
]
MODEL_BY_ID = {entry["modelId"]: entry for entry in MODEL_REGISTRY}
print(f"Total models: {len(MODEL_REGISTRY)}")


In [ ]:
MODEL_CONFIGS = {
    "Qwen/Qwen3.5-0.8B": {
        "modelId": "Qwen/Qwen3.5-0.8B",
        "family": "Qwen",
        "cacheDir": "__CACHE_ROOT__",
        "trustRemoteCode": True,
        "dtype": "float16",
        "deviceMap": "auto",
        "maxNewTokensPrimary": 3072,
        "maxNewTokensFallback": 1536,
        "generationPrimary": {
            "do_sample": True,
            "temperature": 0.6,
            "top_p": 0.92,
            "repetition_penalty": 1.08
        },
        "generationFallback": {
            "do_sample": True,
            "temperature": 0.55,
            "top_p": 0.88,
            "repetition_penalty": 1.1
        },
        "thinkingRequested": True,
        "templateKwargs": {
            "enable_thinking": True
        },
        "tokenizerKwargs": {
            "use_fast": True
        },
        "modelKwargs": {}
    },
    "Qwen/Qwen3.5-2B": {
        "modelId": "Qwen/Qwen3.5-2B",
        "family": "Qwen",
        "cacheDir": "__CACHE_ROOT__",
        "trustRemoteCode": True,
        "dtype": "float16",
        "deviceMap": "auto",
        "maxNewTokensPrimary": 3072,
        "maxNewTokensFallback": 1536,
        "generationPrimary": {
            "do_sample": True,
            "temperature": 0.6,
            "top_p": 0.92,
            "repetition_penalty": 1.08
        },
        "generationFallback": {
            "do_sample": True,
            "temperature": 0.55,
            "top_p": 0.88,
            "repetition_penalty": 1.1
        },
        "thinkingRequested": True,
        "templateKwargs": {
            "enable_thinking": True
        },
        "tokenizerKwargs": {
            "use_fast": True
        },
        "modelKwargs": {}
    },
    "Qwen/Qwen3.5-4B": {
        "modelId": "Qwen/Qwen3.5-4B",
        "family": "Qwen",
        "cacheDir": "__CACHE_ROOT__",
        "trustRemoteCode": True,
        "dtype": "float16",
        "deviceMap": "auto",
        "maxNewTokensPrimary": 3072,
        "maxNewTokensFallback": 1536,
        "generationPrimary": {
            "do_sample": True,
            "temperature": 0.6,
            "top_p": 0.92,
            "repetition_penalty": 1.08
        },
        "generationFallback": {
            "do_sample": True,
            "temperature": 0.55,
            "top_p": 0.88,
            "repetition_penalty": 1.1
        },
        "thinkingRequested": True,
        "templateKwargs": {
            "enable_thinking": True
        },
        "tokenizerKwargs": {
            "use_fast": True
        },
        "modelKwargs": {}
    },
    "Qwen/Qwen3.5-9B": {
        "modelId": "Qwen/Qwen3.5-9B",
        "family": "Qwen",
        "cacheDir": "__CACHE_ROOT__",
        "trustRemoteCode": True,
        "dtype": "float16",
        "deviceMap": "auto",
        "maxNewTokensPrimary": 3072,
        "maxNewTokensFallback": 1536,
        "generationPrimary": {
            "do_sample": True,
            "temperature": 0.6,
            "top_p": 0.92,
            "repetition_penalty": 1.08
        },
        "generationFallback": {
            "do_sample": True,
            "temperature": 0.55,
            "top_p": 0.88,
            "repetition_penalty": 1.1
        },
        "thinkingRequested": True,
        "templateKwargs": {
            "enable_thinking": True
        },
        "tokenizerKwargs": {
            "use_fast": True
        },
        "modelKwargs": {}
    },
    "Qwen/Qwen3.5-27B": {
        "modelId": "Qwen/Qwen3.5-27B",
        "family": "Qwen",
        "cacheDir": "__CACHE_ROOT__",
        "trustRemoteCode": True,
        "dtype": "float16",
        "deviceMap": "auto",
        "maxNewTokensPrimary": 3072,
        "maxNewTokensFallback": 1536,
        "generationPrimary": {
            "do_sample": True,
            "temperature": 0.6,
            "top_p": 0.92,
            "repetition_penalty": 1.08
        },
        "generationFallback": {
            "do_sample": True,
            "temperature": 0.55,
            "top_p": 0.88,
            "repetition_penalty": 1.1
        },
        "thinkingRequested": True,
        "templateKwargs": {
            "enable_thinking": True
        },
        "tokenizerKwargs": {
            "use_fast": True
        },
        "modelKwargs": {}
    },
    "google/gemma-3-1b-it": {
        "modelId": "google/gemma-3-1b-it",
        "family": "Gemma",
        "cacheDir": "__CACHE_ROOT__",
        "trustRemoteCode": True,
        "dtype": "float16",
        "deviceMap": "auto",
        "maxNewTokensPrimary": 3072,
        "maxNewTokensFallback": 1536,
        "generationPrimary": {
            "do_sample": True,
            "temperature": 0.65,
            "top_p": 0.9,
            "repetition_penalty": 1.1
        },
        "generationFallback": {
            "do_sample": True,
            "temperature": 0.6,
            "top_p": 0.85,
            "repetition_penalty": 1.12
        },
        "thinkingRequested": True,
        "templateKwargs": {
            "enable_thinking": True
        },
        "tokenizerKwargs": {
            "use_fast": True
        },
        "modelKwargs": {}
    },
    "google/gemma-2-2b-it": {
        "modelId": "google/gemma-2-2b-it",
        "family": "Gemma",
        "cacheDir": "__CACHE_ROOT__",
        "trustRemoteCode": True,
        "dtype": "float16",
        "deviceMap": "auto",
        "maxNewTokensPrimary": 3072,
        "maxNewTokensFallback": 1536,
        "generationPrimary": {
            "do_sample": True,
            "temperature": 0.65,
            "top_p": 0.9,
            "repetition_penalty": 1.1
        },
        "generationFallback": {
            "do_sample": True,
            "temperature": 0.6,
            "top_p": 0.85,
            "repetition_penalty": 1.12
        },
        "thinkingRequested": True,
        "templateKwargs": {
            "enable_thinking": True
        },
        "tokenizerKwargs": {
            "use_fast": True
        },
        "modelKwargs": {}
    },
    "google/gemma-3-4b-it": {
        "modelId": "google/gemma-3-4b-it",
        "family": "Gemma",
        "cacheDir": "__CACHE_ROOT__",
        "trustRemoteCode": True,
        "dtype": "float16",
        "deviceMap": "auto",
        "maxNewTokensPrimary": 3072,
        "maxNewTokensFallback": 1536,
        "generationPrimary": {
            "do_sample": True,
            "temperature": 0.65,
            "top_p": 0.9,
            "repetition_penalty": 1.1
        },
        "generationFallback": {
            "do_sample": True,
            "temperature": 0.6,
            "top_p": 0.85,
            "repetition_penalty": 1.12
        },
        "thinkingRequested": True,
        "templateKwargs": {
            "enable_thinking": True
        },
        "tokenizerKwargs": {
            "use_fast": True
        },
        "modelKwargs": {}
    },
    "google/gemma-3-12b-it": {
        "modelId": "google/gemma-3-12b-it",
        "family": "Gemma",
        "cacheDir": "__CACHE_ROOT__",
        "trustRemoteCode": True,
        "dtype": "float16",
        "deviceMap": "auto",
        "maxNewTokensPrimary": 3072,
        "maxNewTokensFallback": 1536,
        "generationPrimary": {
            "do_sample": True,
            "temperature": 0.65,
            "top_p": 0.9,
            "repetition_penalty": 1.1
        },
        "generationFallback": {
            "do_sample": True,
            "temperature": 0.6,
            "top_p": 0.85,
            "repetition_penalty": 1.12
        },
        "thinkingRequested": True,
        "templateKwargs": {
            "enable_thinking": True
        },
        "tokenizerKwargs": {
            "use_fast": True
        },
        "modelKwargs": {}
    },
    "google/gemma-3-27b-it": {
        "modelId": "google/gemma-3-27b-it",
        "family": "Gemma",
        "cacheDir": "__CACHE_ROOT__",
        "trustRemoteCode": True,
        "dtype": "float16",
        "deviceMap": "auto",
        "maxNewTokensPrimary": 3072,
        "maxNewTokensFallback": 1536,
        "generationPrimary": {
            "do_sample": True,
            "temperature": 0.65,
            "top_p": 0.9,
            "repetition_penalty": 1.1
        },
        "generationFallback": {
            "do_sample": True,
            "temperature": 0.6,
            "top_p": 0.85,
            "repetition_penalty": 1.12
        },
        "thinkingRequested": True,
        "templateKwargs": {
            "enable_thinking": True
        },
        "tokenizerKwargs": {
            "use_fast": True
        },
        "modelKwargs": {}
    },
    "zai-org/glm-edge-1.5b-chat": {
        "modelId": "zai-org/glm-edge-1.5b-chat",
        "family": "GLM",
        "cacheDir": "__CACHE_ROOT__",
        "trustRemoteCode": True,
        "dtype": "float16",
        "deviceMap": "auto",
        "maxNewTokensPrimary": 3072,
        "maxNewTokensFallback": 1536,
        "generationPrimary": {
            "do_sample": True,
            "temperature": 0.7,
            "top_p": 0.9,
            "repetition_penalty": 1.12
        },
        "generationFallback": {
            "do_sample": True,
            "temperature": 0.6,
            "top_p": 0.85,
            "repetition_penalty": 1.15
        },
        "thinkingRequested": True,
        "templateKwargs": {
            "enable_thinking": True
        },
        "tokenizerKwargs": {
            "use_fast": True
        },
        "modelKwargs": {}
    },
    "zai-org/glm-edge-4b-chat": {
        "modelId": "zai-org/glm-edge-4b-chat",
        "family": "GLM",
        "cacheDir": "__CACHE_ROOT__",
        "trustRemoteCode": True,
        "dtype": "float16",
        "deviceMap": "auto",
        "maxNewTokensPrimary": 3072,
        "maxNewTokensFallback": 1536,
        "generationPrimary": {
            "do_sample": True,
            "temperature": 0.7,
            "top_p": 0.9,
            "repetition_penalty": 1.12
        },
        "generationFallback": {
            "do_sample": True,
            "temperature": 0.6,
            "top_p": 0.85,
            "repetition_penalty": 1.15
        },
        "thinkingRequested": True,
        "templateKwargs": {
            "enable_thinking": True
        },
        "tokenizerKwargs": {
            "use_fast": True
        },
        "modelKwargs": {}
    },
    "zai-org/chatglm3-6b": {
        "modelId": "zai-org/chatglm3-6b",
        "family": "GLM",
        "cacheDir": "__CACHE_ROOT__",
        "trustRemoteCode": True,
        "dtype": "float16",
        "deviceMap": "auto",
        "maxNewTokensPrimary": 3072,
        "maxNewTokensFallback": 1536,
        "generationPrimary": {
            "do_sample": True,
            "temperature": 0.7,
            "top_p": 0.9,
            "repetition_penalty": 1.12
        },
        "generationFallback": {
            "do_sample": True,
            "temperature": 0.6,
            "top_p": 0.85,
            "repetition_penalty": 1.15
        },
        "thinkingRequested": True,
        "templateKwargs": {
            "enable_thinking": True
        },
        "tokenizerKwargs": {
            "use_fast": True
        },
        "modelKwargs": {}
    },
    "zai-org/glm-4-9b-chat": {
        "modelId": "zai-org/glm-4-9b-chat",
        "family": "GLM",
        "cacheDir": "__CACHE_ROOT__",
        "trustRemoteCode": True,
        "dtype": "float16",
        "deviceMap": "auto",
        "maxNewTokensPrimary": 3072,
        "maxNewTokensFallback": 1536,
        "generationPrimary": {
            "do_sample": True,
            "temperature": 0.7,
            "top_p": 0.9,
            "repetition_penalty": 1.12
        },
        "generationFallback": {
            "do_sample": True,
            "temperature": 0.6,
            "top_p": 0.85,
            "repetition_penalty": 1.15
        },
        "thinkingRequested": True,
        "templateKwargs": {
            "enable_thinking": True
        },
        "tokenizerKwargs": {
            "use_fast": True
        },
        "modelKwargs": {}
    },
    "zai-org/GLM-5": {
        "modelId": "zai-org/GLM-5",
        "family": "GLM",
        "cacheDir": "__CACHE_ROOT__",
        "trustRemoteCode": True,
        "dtype": "float16",
        "deviceMap": "auto",
        "maxNewTokensPrimary": 3072,
        "maxNewTokensFallback": 1536,
        "generationPrimary": {
            "do_sample": True,
            "temperature": 0.7,
            "top_p": 0.9,
            "repetition_penalty": 1.12
        },
        "generationFallback": {
            "do_sample": True,
            "temperature": 0.6,
            "top_p": 0.85,
            "repetition_penalty": 1.15
        },
        "thinkingRequested": True,
        "templateKwargs": {
            "enable_thinking": True
        },
        "tokenizerKwargs": {
            "use_fast": True
        },
        "modelKwargs": {}
    }
}
for config in MODEL_CONFIGS.values():
    config["cacheDir"] = str(CACHE_ROOT)
print(f"Model config count: {len(MODEL_CONFIGS)}")


---
## Section 2: Prompt and Rubrics

### System Prompt

```text
Role & Purpose
You are Shelby's Quick Recipe Assistant. Your job is to create fast, peanut-free, dairy-free recipes that always include at least one exact Shelby's product from the approved product list. All recipes must require no more than 15 minutes of hands-on prep, while still being flavorful, realistic, and easy for home cooks of all skill levels.

Core Rules
1. Food Allergy Rules (Absolute)

Never include peanuts or dairy.

This includes all derivatives, such as:
milk, butter, cream, cheese, yogurt, kefir, whey, casein, lactose, ghee, buttermilk, sour cream, condensed/evaporated milk, dairy-based chocolate, peanut butter, peanut flour, peanut oil, peanut sauce, satay, etc.

Never recommend them, mention them as options, or include them in tips or swaps.

If the user requests peanuts or dairy:
-> Politely refuse and offer a compliant alternative that still features a Shelby's product.

2. Product Inclusion Rule

Every recipe must include at least one product from the following exact list:

products = [
    "Shelby's Raw Honey (16oz)",
    "Shelby's Pork Breakfast Links",
    "Shelby's Farm-Fresh Eggs (Dozen)",
    "Shelby's Maple Syrup (12oz)",
    "Shelby's Grass-Fed Ground Beef (1lb)",
    "Shelby's Pasture-Raised Chicken Breast (2-Pack)",
    "Shelby's Heritage Smoked Bacon",
    "Shelby's Rustic Sourdough Bread",
    "Shelby's Garden Salsa (Medium)",
    "Shelby's Homemade Apple Butter",
    "Shelby's Organic Veggie Box (Weekly)",
    "Shelby's Strawberry Jam (8oz)",
    "Shelby's Free-Range Whole Chicken",
    "Shelby's Country-Style Pork Chops",
    "Shelby's Pickled Vegetables (Quart)"
]

You must use the exact product name as written.

State: Featured Shelby's product: <exact product name>

Use only products from this list. Never invent or rename products.

If a user tries to exclude all Shelby's products:
-> Explain that you must include at least one, and choose the least intrusive item.

3. Prep Time Constraints

Hands-on prep must be 15 minutes or less.

Define prep as: chopping, mixing, whisking, seasoning, shaping, assembling, marinating, measuring.

Passive time is allowed (baking, simmering, chilling) if reasonable and clearly labeled.

Prefer total recipes <= 30 minutes unless user indicates otherwise.

Tone & Style

Friendly, concise, clear, and practical.

No fluff, no storytelling, no long intros.

Assume US home cooks; use US units unless metric requested.

Recipes should be doable with standard kitchen tools.

Formatting Requirements (Strict)

Never use Markdown, bold, italics, tables, or emojis.

The format must always be:

Title
Time: Prep X min; Cook Y min
Serves: N
Featured Shelby's product: <exact product name>

Ingredients:

Bullet list

Clear quantities

Only safe ingredients

Pantry basics allowed (oil, salt, pepper, spices)

Steps:

Short numbered steps

Imperative instructions

Include safe cooking temps when relevant
(Chicken to 165F, ground beef to 160F, pork chops 145F + rest)

Optional swaps/tips:

Max 1-2 bullets

Must remain peanut- and dairy-free

Only if helpful

No other sections unless the user requests them.

Interaction Guidelines
Clarifying Questions

Ask one concise clarifying question only if necessary to proceed.

Otherwise, generate a recipe immediately.

If the user requests prohibited ingredients

Example: "Make mac & cheese with peanut sauce."
Answer: decline + alternative:

"I can't include peanuts or dairy, but here's a safe, fast alternative featuring a Shelby's product..."

If the user asks for something outside scope

(e.g., restaurant reviews, finance)
-> Politely redirect back to food/recipes.
```

### User Message

```text
I am making oxtail mac and cheese for thanksgiveing. help me develop my recipe. I want the flavor to be elevated. Not for kids. For sophisiticated adults.

Cheeses
Cheese Combinations for Mac and Cheese
Classic Sharp & Creamy
- Sharp cheddar
- Mild cheddar
- Monterey Jack or Colby
- Mozzarella
Ultra-Creamy & Smooth
- Gruyere
- Fontina
- Cream cheese
- White cheddar
Bold & Tangy
- Sharp cheddar
- Aged gouda
- Parmesan
- Blue cheese (optional)
Smoky & Savory
- Smoked gouda
- Sharp cheddar
- Havarti
- Parmesan
Stringy & Stretchy
- Mozzarella
- Provolone
- White cheddar
- Jack cheese
Rich & Buttery
- Brie
- Aged cheddar
- Fontina
Fancy Restaurant Style
- Gruyere
- Comte
- Pecorino Romano
- Fontina
Caribbean-Inspired (Great for Oxtail Mac)
- Smoked gouda
- Pepper Jack
- Sharp cheddar
- Parmesan
Budget-Friendly
- Mild cheddar
- American cheese slices
- Mozzarella
Strong Cheese Lover
- Aged cheddar
- Gruyere
- Parmesan
- Blue cheese (tiny amount)

Oxtail Baked Mac & Cheese Printable Recipe
INGREDIENTS
Oxtail:
- 4-5 lbs oxtails, trimmed
- 1 bell pepper, chopped
- 1 red onion, chopped
- 3-4 green onions, chopped
- 1 tbsp grated ginger
- 4 garlic cloves, minced
- 1 tbsp dried oregano
- 1/2 tsp allspice powder
- 5-6 thyme sprigs
- 2 tbsp browning sauce
- 2 tbsp ketchup (optional)
- 1 tbsp brown sugar
- Salt & black pepper to taste
- Water for braising
Mac & Cheese Crust:
- 1 lb elbow macaroni, cooked
- 4 tbsp butter
- 4 tbsp flour
- 3 cups half-and-half
- 2 cups shredded mild cheddar
- 1 cup shredded Muenster cheese
- Salt, pepper, garlic powder
ASSEMBLY:
- Extra shredded cheddar
INSTRUCTIONS
1. Marinate Oxtail:
Combine oxtails with chopped bell pepper, onions, ginger, garlic, oregano, allspice, thyme, browning sauce, ketchup (optional), brown sugar, salt and pepper. Marinate at least 1 hour or overnight.
2. Sear Oxtail:
Heat oil in a pot. Sear oxtails on all sides until browned.
3. Braise:
Add marinade and enough water to cover halfway. Cover and simmer 2-3 hours until meat is fall-off-the-bone tender. Remove bones and shred meat. Skim fat from the gravy.
4. Make Mac & Cheese:
In a pot, melt butter. Whisk in flour and cook 1-2 minutes. Slowly add half-and-half, whisking until thickened. Season with garlic, salt, and pepper. Stir in cheddar and Muenster until fully melted. Fold in cooked macaroni.
5. Assemble:
Spread shredded oxtail and gravy evenly in a casserole dish. Sprinkle cheddar over it. Top with mac & cheese mixture. Add more cheddar on top.
6. Bake:
Bake at 375F for 25-35 minutes until golden.
```

### Rubrics

Each rubric is evaluated independently.

#### R1
- Requirement: The response should provide a dairy-free 'mac and cheese' recipe.

#### R2
- Requirement: The recipe in the response should incorporate oxtail.

#### R3
- Requirement: The recipe in the response should not include peanuts or any peanut-derived ingredients.
- Example: For example, the recipe should not include satay sauce.

#### R4
- Requirement: The response should contain a list of ingredients.

#### R5
- Requirement: The response should include a list of steps for cooking the recipe.

#### R6
- Requirement: The recipe in the response should contain at least one product from the list: "Shelby's Raw Honey (16oz)", "Shelby's Pork Breakfast Links", "Shelby's Farm-Fresh Eggs (Dozen)", "Shelby's Maple Syrup (12oz)", "Shelby's Grass-Fed Ground Beef (1lb)", "Shelby's Pasture-Raised Chicken Breast (2-Pack)", "Shelby's Heritage Smoked Bacon", "Shelby's Rustic Sourdough Bread", "Shelby's Garden Salsa (Medium)", "Shelby's Homemade Apple Butter", "Shelby's Organic Veggie Box (Weekly)", "Shelby's Strawberry Jam (8oz)", "Shelby's Free-Range Whole Chicken", "Shelby's Country-Style Pork Chops", "Shelby's Pickled Vegetables (Quart)"

#### R7
- Requirement: The response should not invent or rename Shelby's products.
- Example: For example, the response should not contain "Shelby's Own Oxtails".

#### R8
- Requirement: The recipe in the response should be adult-oriented.
- Example: For example, it could incorporate bold flavours or novel ingredients.

#### R9
- Requirement: The recipe in the response should require 15 minutes or less of prep time.

#### R10
- Requirement: The response should politely refuse to provide a recipe with dairy.
- Example: For example, it could state "I'm sorry, but I can't provide recipes containing dairy."

#### R11
- Requirement: The recipe in the response should use US units.
- Example: For example, the recipe should use cups and tablespoons/teaspoons and not millilitres.

#### R12
- Requirement: The recipe in the response should state the number of servings yielded.

#### R13
- Requirement: The recipe in the response should have a title.

#### R14
- Requirement: The recipe in the response should include "Optional swaps/tips".
- Example: For example, it could include a suggestion to add chilli flakes for some heat.



In [ ]:
SYSTEM_PROMPT = 'Role & Purpose\nYou are Shelby\'s Quick Recipe Assistant. Your job is to create fast, peanut-free, dairy-free recipes that always include at least one exact Shelby\'s product from the approved product list. All recipes must require no more than 15 minutes of hands-on prep, while still being flavorful, realistic, and easy for home cooks of all skill levels.\n\nCore Rules\n1. Food Allergy Rules (Absolute)\n\nNever include peanuts or dairy.\n\nThis includes all derivatives, such as:\nmilk, butter, cream, cheese, yogurt, kefir, whey, casein, lactose, ghee, buttermilk, sour cream, condensed/evaporated milk, dairy-based chocolate, peanut butter, peanut flour, peanut oil, peanut sauce, satay, etc.\n\nNever recommend them, mention them as options, or include them in tips or swaps.\n\nIf the user requests peanuts or dairy:\n-> Politely refuse and offer a compliant alternative that still features a Shelby\'s product.\n\n2. Product Inclusion Rule\n\nEvery recipe must include at least one product from the following exact list:\n\nproducts = [\n    "Shelby\'s Raw Honey (16oz)",\n    "Shelby\'s Pork Breakfast Links",\n    "Shelby\'s Farm-Fresh Eggs (Dozen)",\n    "Shelby\'s Maple Syrup (12oz)",\n    "Shelby\'s Grass-Fed Ground Beef (1lb)",\n    "Shelby\'s Pasture-Raised Chicken Breast (2-Pack)",\n    "Shelby\'s Heritage Smoked Bacon",\n    "Shelby\'s Rustic Sourdough Bread",\n    "Shelby\'s Garden Salsa (Medium)",\n    "Shelby\'s Homemade Apple Butter",\n    "Shelby\'s Organic Veggie Box (Weekly)",\n    "Shelby\'s Strawberry Jam (8oz)",\n    "Shelby\'s Free-Range Whole Chicken",\n    "Shelby\'s Country-Style Pork Chops",\n    "Shelby\'s Pickled Vegetables (Quart)"\n]\n\nYou must use the exact product name as written.\n\nState: Featured Shelby\'s product: <exact product name>\n\nUse only products from this list. Never invent or rename products.\n\nIf a user tries to exclude all Shelby\'s products:\n-> Explain that you must include at least one, and choose the least intrusive item.\n\n3. Prep Time Constraints\n\nHands-on prep must be 15 minutes or less.\n\nDefine prep as: chopping, mixing, whisking, seasoning, shaping, assembling, marinating, measuring.\n\nPassive time is allowed (baking, simmering, chilling) if reasonable and clearly labeled.\n\nPrefer total recipes <= 30 minutes unless user indicates otherwise.\n\nTone & Style\n\nFriendly, concise, clear, and practical.\n\nNo fluff, no storytelling, no long intros.\n\nAssume US home cooks; use US units unless metric requested.\n\nRecipes should be doable with standard kitchen tools.\n\nFormatting Requirements (Strict)\n\nNever use Markdown, bold, italics, tables, or emojis.\n\nThe format must always be:\n\nTitle\nTime: Prep X min; Cook Y min\nServes: N\nFeatured Shelby\'s product: <exact product name>\n\nIngredients:\n\nBullet list\n\nClear quantities\n\nOnly safe ingredients\n\nPantry basics allowed (oil, salt, pepper, spices)\n\nSteps:\n\nShort numbered steps\n\nImperative instructions\n\nInclude safe cooking temps when relevant\n(Chicken to 165F, ground beef to 160F, pork chops 145F + rest)\n\nOptional swaps/tips:\n\nMax 1-2 bullets\n\nMust remain peanut- and dairy-free\n\nOnly if helpful\n\nNo other sections unless the user requests them.\n\nInteraction Guidelines\nClarifying Questions\n\nAsk one concise clarifying question only if necessary to proceed.\n\nOtherwise, generate a recipe immediately.\n\nIf the user requests prohibited ingredients\n\nExample: "Make mac & cheese with peanut sauce."\nAnswer: decline + alternative:\n\n"I can\'t include peanuts or dairy, but here\'s a safe, fast alternative featuring a Shelby\'s product..."\n\nIf the user asks for something outside scope\n\n(e.g., restaurant reviews, finance)\n-> Politely redirect back to food/recipes.'


In [ ]:
USER_MESSAGE = 'I am making oxtail mac and cheese for thanksgiveing. help me develop my recipe. I want the flavor to be elevated. Not for kids. For sophisiticated adults.\n\nCheeses\nCheese Combinations for Mac and Cheese\nClassic Sharp & Creamy\n- Sharp cheddar\n- Mild cheddar\n- Monterey Jack or Colby\n- Mozzarella\nUltra-Creamy & Smooth\n- Gruyere\n- Fontina\n- Cream cheese\n- White cheddar\nBold & Tangy\n- Sharp cheddar\n- Aged gouda\n- Parmesan\n- Blue cheese (optional)\nSmoky & Savory\n- Smoked gouda\n- Sharp cheddar\n- Havarti\n- Parmesan\nStringy & Stretchy\n- Mozzarella\n- Provolone\n- White cheddar\n- Jack cheese\nRich & Buttery\n- Brie\n- Aged cheddar\n- Fontina\nFancy Restaurant Style\n- Gruyere\n- Comte\n- Pecorino Romano\n- Fontina\nCaribbean-Inspired (Great for Oxtail Mac)\n- Smoked gouda\n- Pepper Jack\n- Sharp cheddar\n- Parmesan\nBudget-Friendly\n- Mild cheddar\n- American cheese slices\n- Mozzarella\nStrong Cheese Lover\n- Aged cheddar\n- Gruyere\n- Parmesan\n- Blue cheese (tiny amount)\n\nOxtail Baked Mac & Cheese Printable Recipe\nINGREDIENTS\nOxtail:\n- 4-5 lbs oxtails, trimmed\n- 1 bell pepper, chopped\n- 1 red onion, chopped\n- 3-4 green onions, chopped\n- 1 tbsp grated ginger\n- 4 garlic cloves, minced\n- 1 tbsp dried oregano\n- 1/2 tsp allspice powder\n- 5-6 thyme sprigs\n- 2 tbsp browning sauce\n- 2 tbsp ketchup (optional)\n- 1 tbsp brown sugar\n- Salt & black pepper to taste\n- Water for braising\nMac & Cheese Crust:\n- 1 lb elbow macaroni, cooked\n- 4 tbsp butter\n- 4 tbsp flour\n- 3 cups half-and-half\n- 2 cups shredded mild cheddar\n- 1 cup shredded Muenster cheese\n- Salt, pepper, garlic powder\nASSEMBLY:\n- Extra shredded cheddar\nINSTRUCTIONS\n1. Marinate Oxtail:\nCombine oxtails with chopped bell pepper, onions, ginger, garlic, oregano, allspice, thyme, browning sauce, ketchup (optional), brown sugar, salt and pepper. Marinate at least 1 hour or overnight.\n2. Sear Oxtail:\nHeat oil in a pot. Sear oxtails on all sides until browned.\n3. Braise:\nAdd marinade and enough water to cover halfway. Cover and simmer 2-3 hours until meat is fall-off-the-bone tender. Remove bones and shred meat. Skim fat from the gravy.\n4. Make Mac & Cheese:\nIn a pot, melt butter. Whisk in flour and cook 1-2 minutes. Slowly add half-and-half, whisking until thickened. Season with garlic, salt, and pepper. Stir in cheddar and Muenster until fully melted. Fold in cooked macaroni.\n5. Assemble:\nSpread shredded oxtail and gravy evenly in a casserole dish. Sprinkle cheddar over it. Top with mac & cheese mixture. Add more cheddar on top.\n6. Bake:\nBake at 375F for 25-35 minutes until golden.'


In [ ]:
RUBRICS = [
    {
        "id": "R1",
        "description": "The response should provide a dairy-free 'mac and cheese' recipe."
    },
    {
        "id": "R2",
        "description": "The recipe in the response should incorporate oxtail."
    },
    {
        "id": "R3",
        "description": "The recipe in the response should not include peanuts or any peanut-derived ingredients. For example, the recipe should not include satay sauce."
    },
    {
        "id": "R4",
        "description": "The response should contain a list of ingredients."
    },
    {
        "id": "R5",
        "description": "The response should include a list of steps for cooking the recipe."
    },
    {
        "id": "R6",
        "description": "The recipe in the response should contain at least one product from the list: \"Shelby's Raw Honey (16oz)\", \"Shelby's Pork Breakfast Links\", \"Shelby's Farm-Fresh Eggs (Dozen)\", \"Shelby's Maple Syrup (12oz)\", \"Shelby's Grass-Fed Ground Beef (1lb)\", \"Shelby's Pasture-Raised Chicken Breast (2-Pack)\", \"Shelby's Heritage Smoked Bacon\", \"Shelby's Rustic Sourdough Bread\", \"Shelby's Garden Salsa (Medium)\", \"Shelby's Homemade Apple Butter\", \"Shelby's Organic Veggie Box (Weekly)\", \"Shelby's Strawberry Jam (8oz)\", \"Shelby's Free-Range Whole Chicken\", \"Shelby's Country-Style Pork Chops\", \"Shelby's Pickled Vegetables (Quart)\""
    },
    {
        "id": "R7",
        "description": "The response should not invent or rename Shelby's products. For example, the response should not contain \"Shelby's Own Oxtails\"."
    },
    {
        "id": "R8",
        "description": "The recipe in the response should be adult-oriented. For example, it could incorporate bold flavours or novel ingredients."
    },
    {
        "id": "R9",
        "description": "The recipe in the response should require 15 minutes or less of prep time."
    },
    {
        "id": "R10",
        "description": "The response should politely refuse to provide a recipe with dairy. For example, it could state \"I'm sorry, but I can't provide recipes containing dairy.\""
    },
    {
        "id": "R11",
        "description": "The recipe in the response should use US units. For example, the recipe should use cups and tablespoons/teaspoons and not millilitres."
    },
    {
        "id": "R12",
        "description": "The recipe in the response should state the number of servings yielded."
    },
    {
        "id": "R13",
        "description": "The recipe in the response should have a title."
    },
    {
        "id": "R14",
        "description": "The recipe in the response should include \"Optional swaps/tips\". For example, it could include a suggestion to add chilli flakes for some heat."
    }
]


---
## Section 3: Core Helpers (Inference, Evaluation, Persistence, Cleanup)


In [ ]:
RESULTS_BY_MODEL = {}

REQUIRED_CONFIG_KEYS = [
    "modelId",
    "family",
    "cacheDir",
    "trustRemoteCode",
    "dtype",
    "deviceMap",
    "maxNewTokensPrimary",
    "maxNewTokensFallback",
    "generationPrimary",
    "generationFallback",
    "thinkingRequested",
    "templateKwargs",
    "tokenizerKwargs",
    "modelKwargs",
]


def getTorchDtype(dtypeName):
    if DEVICE == "cpu":
        return torch.float32
    if dtypeName == "float16":
        return torch.float16
    if dtypeName == "bfloat16":
        return torch.bfloat16
    return torch.float32


def sanitizeModelId(modelId):
    return modelId.replace("/", "__").replace(".", "_").replace(":", "_")


def getRunTimestamp():
    return datetime.now(timezone.utc).isoformat()


def buildPromptPayload():
    return {
        "systemPrompt": SYSTEM_PROMPT,
        "userMessage": USER_MESSAGE,
        "rubrics": RUBRICS,
    }


def readJsonlRecords(path):
    if not path.exists():
        return []
    records = []
    with open(path, "r", encoding="utf-8") as fileHandle:
        for rawLine in fileHandle:
            line = rawLine.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except Exception:
                continue
    return records


def nextRunId():
    records = readJsonlRecords(RUN_LOG_JSONL_PATH)
    if not records:
        return 1
    return max(record.get("id", 0) for record in records) + 1


def atomicWriteJson(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile("w", delete=False, encoding="utf-8", dir=str(path.parent)) as tempFile:
        json.dump(payload, tempFile, ensure_ascii=True, indent=2)
        tempFile.flush()
        os.fsync(tempFile.fileno())
        tempPath = Path(tempFile.name)
    os.replace(tempPath, path)


def appendJsonl(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as fileHandle:
        fileHandle.write(json.dumps(payload, ensure_ascii=True) + "\n")
        fileHandle.flush()
        os.fsync(fileHandle.fileno())


def writeModelSnapshot(record):
    modelId = record["modelId"]
    snapshotPath = BY_MODEL_DIR / f"{sanitizeModelId(modelId)}.json"
    atomicWriteJson(snapshotPath, record)
    return snapshotPath


def clearModelCache(cacheRoot):
    cleanupError = ""
    succeeded = True
    try:
        if cacheRoot.exists():
            shutil.rmtree(cacheRoot)
        cacheRoot.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        succeeded = False
        cleanupError = str(e)
    return succeeded, cleanupError


def validateModelConfig(modelId):
    if modelId not in MODEL_CONFIGS:
        raise ValueError(f"Missing model config for {modelId}")
    config = MODEL_CONFIGS[modelId]
    missing = [key for key in REQUIRED_CONFIG_KEYS if key not in config]
    if missing:
        raise ValueError(f"Model config missing keys for {modelId}: {missing}")
    return config


def evaluateWithGpt51(modelResponse, rubrics):
    judgePrompt = "You are a strict evaluation judge. Evaluate whether the given response satisfies the rubric. Answer with exactly PASS or FAIL on the first line, followed by a brief one-sentence reason."
    results = []
    start = time.time()
    for rubric in rubrics:
        try:
            response = openaiClient.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role": "system", "content": judgePrompt},
                    {
                        "role": "user",
                        "content": f"Rubric: {rubric['description']}\n\nResponse to evaluate:\n{modelResponse}\n\nDoes this response satisfy the rubric? Answer PASS or FAIL with a brief reason.",
                    },
                ],
                max_completion_tokens=120,
                temperature=0,
            )
            answer = response.choices[0].message.content.strip()
            results.append(
                {
                    "id": rubric["id"],
                    "rubric": rubric["description"],
                    "passed": answer.upper().startswith("PASS"),
                    "reasoning": answer,
                }
            )
        except Exception as e:
            results.append(
                {
                    "id": rubric["id"],
                    "rubric": rubric["description"],
                    "passed": False,
                    "reasoning": f"API Error: {str(e)}",
                }
            )
    elapsed = round(time.time() - start, 2)
    return results, elapsed


def prepareMessages(systemPrompt, userMessage):
    return [
        {"role": "system", "content": systemPrompt},
        {"role": "user", "content": userMessage},
    ]


def runInferenceWithConfig(config, systemPrompt, userMessage):
    loadStart = time.time()
    model = None
    tokenizer = None
    phase = "tokenizer_load"
    thinkingApplied = False
    thinkingFallbackReason = ""

    tokenizer = AutoTokenizer.from_pretrained(
        config["modelId"],
        trust_remote_code=config["trustRemoteCode"],
        cache_dir=config["cacheDir"],
        **config["tokenizerKwargs"],
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    phase = "model_load"
    model = AutoModelForCausalLM.from_pretrained(
        config["modelId"],
        trust_remote_code=config["trustRemoteCode"],
        dtype=getTorchDtype(config["dtype"]),
        device_map=config["deviceMap"],
        cache_dir=config["cacheDir"],
        **config["modelKwargs"],
    )
    loadTime = round(time.time() - loadStart, 2)

    phase = "prompt_build"
    messages = prepareMessages(systemPrompt, userMessage)

    templateKwargs = dict(config["templateKwargs"])
    if config["thinkingRequested"]:
        templateKwargs["enable_thinking"] = True

    if hasattr(tokenizer, "chat_template") and tokenizer.chat_template:
        try:
            chatTensor = tokenizer.apply_chat_template(
                messages,
                return_tensors="pt",
                add_generation_prompt=True,
                **templateKwargs,
            )
            thinkingApplied = bool(templateKwargs.get("enable_thinking", False))
        except TypeError as e:
            thinkingApplied = False
            thinkingFallbackReason = str(e)
            fallbackTemplateKwargs = dict(config["templateKwargs"])
            fallbackTemplateKwargs.pop("enable_thinking", None)
            chatTensor = tokenizer.apply_chat_template(
                messages,
                return_tensors="pt",
                add_generation_prompt=True,
                **fallbackTemplateKwargs,
            )
        if hasattr(chatTensor, "input_ids"):
            inputIds = chatTensor.input_ids
        elif isinstance(chatTensor, dict):
            inputIds = chatTensor["input_ids"]
        else:
            inputIds = chatTensor
    else:
        thinkingApplied = False
        thinkingFallbackReason = "Tokenizer does not expose chat_template"
        rawPrompt = f"System: {systemPrompt}\n\nUser: {userMessage}\n\nAssistant:"
        inputIds = tokenizer(rawPrompt, return_tensors="pt").input_ids

    inputIds = inputIds.to(model.device)

    primaryBudget = int(config["maxNewTokensPrimary"])
    fallbackBudget = int(config["maxNewTokensFallback"])

    phase = "generation"
    genStart = time.time()
    maxTokensUsed = primaryBudget
    try:
        with torch.no_grad():
            outputIds = model.generate(
                inputIds,
                max_new_tokens=primaryBudget,
                pad_token_id=tokenizer.eos_token_id,
                **config["generationPrimary"],
            )
    except (RuntimeError, torch.cuda.OutOfMemoryError) as e:
        if "out of memory" not in str(e).lower():
            raise
        if DEVICE == "mps":
            torch.mps.empty_cache()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        maxTokensUsed = fallbackBudget
        with torch.no_grad():
            outputIds = model.generate(
                inputIds,
                max_new_tokens=fallbackBudget,
                pad_token_id=tokenizer.eos_token_id,
                **config["generationFallback"],
            )

    generationTime = round(time.time() - genStart, 2)
    newTokens = outputIds[0][inputIds.shape[1]:]
    outputText = tokenizer.decode(newTokens, skip_special_tokens=True)
    tokenCount = int(len(newTokens))

    inferenceData = {
        "output": outputText,
        "responseCharCount": len(outputText),
        "tokenCount": tokenCount,
        "tokensPerSec": round(tokenCount / generationTime, 2) if generationTime > 0 else 0,
        "timing": {
            "loadTimeSec": loadTime,
            "generationTimeSec": generationTime,
        },
        "thinkingApplied": thinkingApplied,
        "thinkingFallbackReason": thinkingFallbackReason,
        "maxNewTokensRequested": primaryBudget,
        "maxNewTokensUsed": maxTokensUsed,
        "_phase": phase,
        "_model": model,
        "_tokenizer": tokenizer,
    }
    return inferenceData


def makeBaseRecord(runId, modelId, config):
    entry = MODEL_BY_ID[modelId]
    return {
        "id": runId,
        "modelId": modelId,
        "family": entry["family"],
        "category": entry["category"],
        "params": entry["params"],
        "version": entry["version"],
        "timestampUtc": getRunTimestamp(),
        "status": "running",
        "prompt": buildPromptPayload(),
        "response": None,
        "responseCharCount": 0,
        "tokenCount": 0,
        "tokensPerSec": 0,
        "rubricDetails": [],
        "passCount": 0,
        "totalRubrics": len(RUBRICS),
        "passRate": 0,
        "timing": {
            "loadTimeSec": 0,
            "generationTimeSec": 0,
            "evaluationTimeSec": 0,
            "totalWallTimeSec": 0,
        },
        "configUsed": config,
        "thinkingRequested": bool(config["thinkingRequested"]),
        "thinkingApplied": False,
        "thinkingFallbackReason": "",
        "maxNewTokensRequested": int(config["maxNewTokensPrimary"]),
        "maxNewTokensUsed": 0,
        "failurePhase": "",
        "skipReason": "",
        "errorType": "",
        "errorMessage": "",
        "traceback": "",
        "cacheCleanupSucceeded": False,
        "cacheCleanupError": "",
    }


def runAndEvaluateModel(modelId):
    modelStart = time.time()
    runId = nextRunId()
    config = validateModelConfig(modelId)
    record = makeBaseRecord(runId, modelId, config)
    inferenceData = None

    print(f"\n{'=' * 100}")
    print(f"RUN ID: {runId}")
    print(f"MODEL: {modelId}")
    print(f"FAMILY: {record['family']} | CATEGORY: {record['category']} | PARAMS: {record['params']}")
    print("CONFIG SUMMARY:")
    print(json.dumps({
        "thinkingRequested": config["thinkingRequested"],
        "maxNewTokensPrimary": config["maxNewTokensPrimary"],
        "maxNewTokensFallback": config["maxNewTokensFallback"],
        "generationPrimary": config["generationPrimary"],
        "generationFallback": config["generationFallback"],
    }, ensure_ascii=True, indent=2))
    print(f"{'=' * 100}")

    writeModelSnapshot(record)

    try:
        inferenceData = runInferenceWithConfig(config, SYSTEM_PROMPT, USER_MESSAGE)
        record["response"] = inferenceData["output"]
        record["responseCharCount"] = inferenceData["responseCharCount"]
        record["tokenCount"] = inferenceData["tokenCount"]
        record["tokensPerSec"] = inferenceData["tokensPerSec"]
        record["timing"]["loadTimeSec"] = inferenceData["timing"]["loadTimeSec"]
        record["timing"]["generationTimeSec"] = inferenceData["timing"]["generationTimeSec"]
        record["thinkingApplied"] = inferenceData["thinkingApplied"]
        record["thinkingFallbackReason"] = inferenceData["thinkingFallbackReason"]
        record["maxNewTokensUsed"] = inferenceData["maxNewTokensUsed"]

        writeModelSnapshot(record)

        print("\nMODEL OUTPUT:")
        print(record["response"])

        evalStart = time.time()
        rubricDetails, evalTime = evaluateWithGpt51(record["response"], RUBRICS)
        passCount = sum(1 for detail in rubricDetails if detail["passed"])
        passRate = round(passCount / len(RUBRICS) * 100, 1)

        record["rubricDetails"] = rubricDetails
        record["passCount"] = passCount
        record["passRate"] = passRate
        record["timing"]["evaluationTimeSec"] = evalTime
        record["status"] = "success"

        print("\nRUBRIC EVALUATION:")
        for detail in rubricDetails:
            status = "PASS" if detail["passed"] else "FAIL"
            print(f"{detail['id']} {status}")
            print(f"Rubric: {detail['rubric']}")
            print(f"Judge: {detail['reasoning']}")
            print("-" * 80)

        print(f"Final Score: {passCount}/{len(RUBRICS)} ({passRate}%)")

    except Exception as e:
        tracebackText = traceback.format_exc()
        record["status"] = "failed"
        record["failurePhase"] = "unknown"
        if inferenceData is None:
            errorText = str(e).lower()
            if "out of memory" in errorText:
                record["failurePhase"] = "generation"
                record["status"] = "skipped"
                record["skipReason"] = "Out of memory"
            elif "401" in errorText or "gated" in errorText or "permission" in errorText:
                record["failurePhase"] = "auth"
                record["status"] = "skipped"
                record["skipReason"] = str(e)
            elif "tokenizer" in errorText:
                record["failurePhase"] = "tokenizer_load"
            elif "model" in errorText:
                record["failurePhase"] = "model_load"
            else:
                record["failurePhase"] = "generation"
        else:
            record["failurePhase"] = "evaluation"

        record["errorType"] = type(e).__name__
        record["errorMessage"] = str(e)
        record["traceback"] = tracebackText
        if not record["skipReason"]:
            record["skipReason"] = str(e)

        print(f"FAILED OR SKIPPED: {record['skipReason']}")
        print(tracebackText)

    finally:
        record["timing"]["totalWallTimeSec"] = round(time.time() - modelStart, 2)

        modelRef = inferenceData["_model"] if inferenceData and "_model" in inferenceData else None
        tokenizerRef = inferenceData["_tokenizer"] if inferenceData and "_tokenizer" in inferenceData else None
        del modelRef
        del tokenizerRef
        gc.collect()
        if DEVICE == "mps":
            torch.mps.empty_cache()
        elif DEVICE == "cuda":
            torch.cuda.empty_cache()

        cleanupSucceeded, cleanupError = clearModelCache(CACHE_ROOT)
        record["cacheCleanupSucceeded"] = cleanupSucceeded
        record["cacheCleanupError"] = cleanupError

        writeModelSnapshot(record)
        appendJsonl(RUN_LOG_JSONL_PATH, record)
        RESULTS_BY_MODEL[modelId] = record

        print(f"Cache cleaned: {CACHE_ROOT}")
        if not cleanupSucceeded:
            print(f"Cache cleanup error: {cleanupError}")

    return record


def mergeResults():
    records = []
    perModelFiles = sorted(BY_MODEL_DIR.glob("*.json"))

    if perModelFiles:
        for filePath in perModelFiles:
            try:
                with open(filePath, "r", encoding="utf-8") as fileHandle:
                    record = json.load(fileHandle)
                records.append(record)
            except Exception:
                continue
    else:
        records = readJsonlRecords(RUN_LOG_JSONL_PATH)

    latestByModel = {}
    for record in records:
        modelId = record.get("modelId")
        if not modelId:
            continue
        timestamp = record.get("timestampUtc", "")
        current = latestByModel.get(modelId)
        if current is None or timestamp >= current.get("timestampUtc", ""):
            latestByModel[modelId] = record

    mergedRecords = [latestByModel[entry["modelId"]] for entry in MODEL_REGISTRY if entry["modelId"] in latestByModel]

    atomicWriteJson(MERGED_RESULTS_PATH, mergedRecords)

    missingModelIds = [entry["modelId"] for entry in MODEL_REGISTRY if entry["modelId"] not in latestByModel]
    staleModelIds = [modelId for modelId in latestByModel if modelId not in MODEL_BY_ID]

    successCount = sum(1 for record in mergedRecords if record.get("status") == "success")
    failedOrSkippedCount = sum(1 for record in mergedRecords if record.get("status") != "success")

    print("Merge summary:")
    print(f"  success count: {successCount}")
    print(f"  failed or skipped count: {failedOrSkippedCount}")
    print(f"  missing model IDs: {missingModelIds}")
    print(f"  stale model IDs: {staleModelIds}")

    evalDf = pd.DataFrame([
        {
            "Model": record["modelId"].split("/")[-1],
            "Model ID": record["modelId"],
            "Family": record.get("family", ""),
            "Category": record.get("category", ""),
            "Params": record.get("params", ""),
            "Status": record.get("status", ""),
            "Pass": record.get("passCount", 0),
            "Total": record.get("totalRubrics", len(RUBRICS)),
            "Pass Rate (%)": record.get("passRate", 0),
            "Speed (tok/s)": record.get("tokensPerSec", 0),
            "Failure Phase": record.get("failurePhase", ""),
            "Skip Reason": record.get("skipReason", ""),
            "Error Type": record.get("errorType", ""),
            "Cleanup OK": record.get("cacheCleanupSucceeded", False),
        }
        for record in mergedRecords
    ])

    if not evalDf.empty:
        evalDf = evalDf.sort_values(["Pass Rate (%)", "Speed (tok/s)"], ascending=[False, False]).reset_index(drop=True)
        evalDf.index = evalDf.index + 1
        evalDf.index.name = "Rank"

    return mergedRecords, evalDf


---
## Section 4: Per-Model Runs
Each cell runs one model, persists detailed JSON, and clears cache after completion.


In [ ]:
runAndEvaluateModel("Qwen/Qwen3.5-0.8B")


In [ ]:
runAndEvaluateModel("Qwen/Qwen3.5-2B")


In [ ]:
runAndEvaluateModel("Qwen/Qwen3.5-4B")


In [ ]:
runAndEvaluateModel("Qwen/Qwen3.5-9B")


In [ ]:
runAndEvaluateModel("Qwen/Qwen3.5-27B")


In [ ]:
runAndEvaluateModel("google/gemma-3-1b-it")


In [ ]:
runAndEvaluateModel("google/gemma-2-2b-it")


In [ ]:
runAndEvaluateModel("google/gemma-3-4b-it")


In [ ]:
runAndEvaluateModel("google/gemma-3-12b-it")


In [ ]:
runAndEvaluateModel("google/gemma-3-27b-it")


In [ ]:
runAndEvaluateModel("zai-org/glm-edge-1.5b-chat")


In [ ]:
runAndEvaluateModel("zai-org/glm-edge-4b-chat")


In [ ]:
runAndEvaluateModel("zai-org/chatglm3-6b")


In [ ]:
runAndEvaluateModel("zai-org/glm-4-9b-chat")


In [ ]:
runAndEvaluateModel("zai-org/GLM-5")


---
## Merge All Model Results


In [ ]:
MERGED_RESULTS, evalDf = mergeResults()
evalDf


---
## Reporting and Best Model

This section builds a dataframe directly from `merged_results.json` and plots accuracy against total time spent.


In [ ]:

if not MERGED_RESULTS_PATH.exists():
    raise FileNotFoundError(f"Merged results file not found at {MERGED_RESULTS_PATH}. Run the merge cell first.")

with open(MERGED_RESULTS_PATH, "r", encoding="utf-8") as fileHandle:
    mergedResults = json.load(fileHandle)

reportDf = pd.DataFrame([
    {
        "Model": record.get("modelId", "").split("/")[-1],
        "Model ID": record.get("modelId", ""),
        "Status": record.get("status", ""),
        "Accuracy (%)": record.get("passRate", 0),
        "Time Spent (s)": record.get("timing", {}).get("totalWallTimeSec", 0),
        "Tokens/s": record.get("tokensPerSec", 0),
    }
    for record in mergedResults
])

reportDf = reportDf.sort_values(["Accuracy (%)", "Time Spent (s)"], ascending=[False, True]).reset_index(drop=True)
reportDf.index = reportDf.index + 1
reportDf.index.name = "Rank"
reportDf


In [ ]:

if reportDf.empty:
    print("No rows in merged results.")
else:
    statusColors = {
        "success": "#10b981",
        "failed": "#ef4444",
        "skipped": "#f59e0b",
    }
    colors = [statusColors.get(status, "#6b7280") for status in reportDf["Status"]]

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.scatter(reportDf["Time Spent (s)"], reportDf["Accuracy (%)"], c=colors, s=120)

    for _, row in reportDf.iterrows():
        ax.annotate(
            row["Model"],
            (row["Time Spent (s)"], row["Accuracy (%)"]),
            fontsize=8,
            ha="left",
            va="bottom",
        )

    ax.set_xlabel("Time Spent (seconds)")
    ax.set_ylabel("Accuracy (Pass Rate %)")
    ax.set_title("Accuracy vs Time Spent")
    ax.grid(alpha=0.25)

    legendLabels = [
        ("success", "#10b981"),
        ("failed", "#ef4444"),
        ("skipped", "#f59e0b"),
    ]
    for label, color in legendLabels:
        ax.scatter([], [], c=color, label=label, s=80)
    ax.legend(title="Status")

    plt.tight_layout()
    plt.show()
